### Imports

In [ ]:
import torch
import pandas as pd
import plotly.express as px
import numpy as np

from transformer_lens import HookedTransformer
from huggingface_hub import login
from dotenv import load_dotenv

import os
import random
import re

/home/grenadinu/University/Article/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.0.1
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


AttributeError: module transformers has no attribute TRANSFORMERS_CACHE

In [ ]:
load_dotenv()
login(token=os.getenv("HF_TOKEN"))

In [ ]:
models = [
    "pythia-70m",
    "gpt2-small",
    "gpt2-medium",
    "gemma-2b",
]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = models[3]

print(f"Loading model {model_name} on {device}...")

Loading model gpt2-medium on cuda...


In [ ]:
model = HookedTransformer.from_pretrained(model_name, device=device)

Loading weights: 100%|██████████| 292/292 [00:01<00:00, 243.28it/s]
/home/grenadinu/University/Article/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:371: UserWarning: Found GPU0 NVIDIA GeForce GTX 1050 Ti which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.6
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/grenadinu/University/Article/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:489: UserWarning: 
NVIDIA GeForce

Loaded pretrained model gpt2-medium into HookedTransformer


### Utils

In [ ]:
def inject_noise(text, noise_type="markup"):
    """Injects specific types of noise to test layer robustness."""
    if noise_type == "markup":
        tags = ["<div>", "<b>", "<span>", "<p>"]
        tag = random.choice(tags)
        return f"{tag}{text}{tag.replace('<', '</')}"

    elif noise_type == "symbols":
        syms = ["!!!", "@@@", "***", "###"]
        s = random.choice(syms)
        return f"{s} {text} {s}"

    elif noise_type == "marketing":
        phrases = ["[BEST SELLER]", "[LIMITED TIME]", "FREE SHIPPING!", "100% GENUINE"]
        return f"{random.choice(phrases)} {text}"
        
    return text

In [ ]:
def get_sufficiency_layer(model, prompt, target_token_str, threshold=0.90, min_final_confidence=0.10):
    """
    Calculates L_suff: The earliest layer where the target token's probability 
    reaches the threshold % of its final layer probability.
    """
    # 1. Get the single token ID for the target
    # try:
    #     target_token_id = model.to_single_token(target_token_str)
    # except Exception as e:
    #     print("Broken string", e)
    #     # If the target string breaks into multiple tokens, this skips it for simplicity
    #     return None, 0.0, False 
    tokens = model.to_tokens(target_token_str, prepend_bos=False)
    if tokens.shape[-1] > 1:
        # Use the first token of the target word for the analysis
        target_token_id = tokens[0, 0].item() 
    else:
        target_token_id = tokens.item()

    # 2. Run cache
    logits, cache = model.run_with_cache(prompt)
    
    # 3. Extract accumulated residual stream and project to vocabulary
    resid_stack = cache.accumulated_resid(incl_mid=False)
    final_pos_resid = resid_stack[:, 0, -1, :] 
    layer_logits = model.unembed(model.ln_final(final_pos_resid))
    layer_probs = torch.softmax(layer_logits, dim=-1)
    
    # 4. Extract probability trajectory for the specific target
    target_probs = layer_probs[:, target_token_id].detach().cpu().numpy()
    final_prob = target_probs[-1]
    
    # 5. Success Check: Did the model actually solve the task at the end?
    if final_prob < min_final_confidence:
        # print("Model failed the task: ", final_prob)
        return None, final_prob, False # Model failed the task, discard this sample
        
    # 6. Calculate L_suff
    target_threshold = threshold * final_prob
    l_suff = len(target_probs) - 1 # Default to final layer
    
    for layer_idx, prob in enumerate(target_probs):
        if prob >= target_threshold:
            l_suff = layer_idx
            break
    
    # print("Model successfully completed the task: ", prob)
    return l_suff, final_prob, True

In [ ]:
def run_experiment(model, df, sample, threshold):
    print("Loading dataset...")

    try:
        raw_products = df["product_name"].dropna().unique().tolist()[:200]
    except FileNotFoundError:
        print("CSV not found. Generating synthetic dataset for demonstration...")
        raw_products = [
            "Adidas Ultraboost 22", "Sony PlayStation 5", "Apple iPhone 14",
            "Samsung Galaxy S23", "Nike Air Max", "Logitech MX Master 3",
            "Bose QuietComfort 45", "Dell XPS 13", "Nintendo Switch OLED",
            "Dyson V15 Detect"
        ]

    results = []
    noise_categories = ["markup", "symbols", "marketing"]
    
    print(f"Starting batch processing across {len(raw_products)} samples...")
    with torch.no_grad():
      for category in noise_categories:
          print(f"Processing category: {category}")
          success_count = 0
          
          for original in raw_products:
              # Generate the dirty input
              noisy_input = inject_noise(original, category)
              
              # The target is the first main token of the original product name
              # Prepend a space because tokenization usually includes leading spaces
              target_token_str = " " + original.split()[0] 
              
              # Few-shot prompt to force the model into extraction mode
              prompt = (
                  "Task: Extract the core brand name.\n"
                  "**Example:** Input: <p>NIKE shoes cheap</p> | Clean: Nike\n"
                  f"Input: {noisy_input} | Clean:"
              )
              
              l_suff, final_prob, is_success = get_sufficiency_layer(
                  model, 
                  prompt, 
                  target_token_str, 
                  threshold=threshold
              )
              
              if is_success:
                  results.append({
                      "Original": original,
                      "Noisy_Input": noisy_input,
                      "Target_Token": target_token_str,
                      "Noise_Category": category,
                      "L_suff": l_suff,
                      "Final_Confidence": final_prob
                  })
                  success_count += 1
                  
          print(f"  -> Successfully evaluated {success_count}/{len(raw_products)} samples.")

    if not results:
        print("No successful evaluations to plot. Check your prompt or target tokens.")
        return

    results_df = pd.DataFrame(results)
    
    # Save to CSV for your article's appendix or further analysis
    results_df.to_csv(f"../data/03-results/layer_sufficiency_results_{sample}_{threshold}.csv", index=False)
    print("\nResults saved to 'layer_sufficiency_results.csv'")
    
    # Generate the Interactive Plotly Chart
    fig = px.box(
        results_df, 
        x="Noise_Category", 
        y="L_suff", 
        color="Noise_Category",
        title="Computational Depth Required per Noise Type (Layer Sufficiency)",
        labels={
            "Noise_Category": "Type of Injected Noise",
            "L_suff": "Sufficiency Layer (L_suff) at 90% Confidence"
        },
        template="plotly_white",
        points="all" # Shows individual data points alongside the box
    )
    
    fig.update_layout(
        yaxis=dict(tickmode='linear', tick0=0, dtick=1)
    )
    
    fig.show()

### Dataset Load

In [ ]:
df = pd.read_csv("../data/02-processed/features.csv")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1465 entries, 0 to 1464
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   product_id           1465 non-null   str  
 1   product_name         1465 non-null   str  
 2   category             1465 non-null   str  
 3   discounted_price     1465 non-null   str  
 4   actual_price         1465 non-null   str  
 5   discount_percentage  1465 non-null   str  
 6   rating               1465 non-null   str  
 7   rating_count         1463 non-null   str  
 8   about_product        1465 non-null   str  
 9   user_id              1465 non-null   str  
 10  user_name            1465 non-null   str  
 11  review_id            1465 non-null   str  
 12  review_title         1465 non-null   str  
 13  review_content       1465 non-null   str  
 14  img_link             1465 non-null   str  
 15  product_link         1465 non-null   str  
dtypes: str(16)
memory usage: 4.7 MB


In [ ]:
df.head(1)

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,₹329,₹699,53%,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,₹154,₹399,61%,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


### LLM Layers Analysis

In [ ]:
sample = 100
df = df.sample(n=sample, random_state=42).reset_index(drop=True)
threshold = 0.9

In [ ]:
run_experiment(model, df, sample, threshold)